**Sparse overcomplete autoencoder for representing features learned by the mlp.**

In [4]:
import torch.nn as nn
import torch

**Sparse overcomplete autoencoder definition**
- Overcomplete because featuresin mlps are represented in superpositions (more features than neurons) 
    -to represent all the features then, the dictionary must be overcomplete, also because its sparse

In [5]:
class SparseOvercompleteAutoencoder(nn.Module):
    def __init__(self, d_model, num_features):
        super(SparseOvercompleteAutoencoder, self).__init__()
        self.d_model = d_model
        self.num_features = num_features

        self.W_enc = nn.Linear(d_model, num_features)
        self.W_dec = nn.Linear(num_features, d_model)
        
    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

    def encode(self, x):
        z = torch.relu(self.W_enc(x-self.W_dec.bias))
        return z
    
    def decode(self, z):
        x_hat = self.W_dec(z)
        return x_hat

    def normalize_decoder_(self):
        weight = self.W_dec.weight.data
        weight /= weight.norm(p=2, dim=0, keepdim=True).clamp(min=1e-8)
    
    def loss(self, x, x_hat, z, l1_coeff = 0.01):
        recon = (x_hat - x).pow(2).mean()
        sparsity = z.abs().sum(dim=1).mean()
        return recon + l1_coeff * sparsity
        


**Training sparse autoencoder**

In [7]:
activations = torch.load("activations for 128 model size/encoder_mlp_acts.pt")

sae = SparseOvercompleteAutoencoder(128, 1024)
sae.normalize_decoder_()
opt = torch.optim.Adam(sae.parameters(), lr=1e-3)

for step in range(2000):
    batch = activations[torch.randint(0, activations.size(0), (512,))]
    x_hat, z = sae(batch)
    loss = sae.loss(batch, x_hat, z)
    opt.zero_grad()
    loss.backward()
    opt.step()
    sae.normalize_decoder_()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item()}")

Step 0, Loss: 16.54859161376953
Step 100, Loss: 0.8413006663322449
Step 200, Loss: 0.5112171173095703
Step 300, Loss: 0.4172818660736084
Step 400, Loss: 0.38732972741127014
Step 500, Loss: 0.37722188234329224
Step 600, Loss: 0.37362316250801086
Step 700, Loss: 0.36443978548049927
Step 800, Loss: 0.3526598811149597
Step 900, Loss: 0.3465498387813568
Step 1000, Loss: 0.3440939784049988
Step 1100, Loss: 0.33014070987701416
Step 1200, Loss: 0.3307987153530121
Step 1300, Loss: 0.32878565788269043
Step 1400, Loss: 0.32315051555633545
Step 1500, Loss: 0.32409894466400146
Step 1600, Loss: 0.31512826681137085
Step 1700, Loss: 0.3220604658126831
Step 1800, Loss: 0.3146568834781647
Step 1900, Loss: 0.3138360381126404
